# Deformation visualization

Runs ElasticDino inference and draws each sampling field on the input image, in the order the upscaler applies them.

The wide figure has one column per moment:

- **Top:** the photo warped by the same fields that resample the features.
- **Middle:** a lattice warped the same way. Red lines started horizontal, cyan lines started vertical.
- **Bottom:** that step's displacement on the guidance image. Hue is direction, white is no shift.

The second image is the composed field. Every block and every nearest upsample is folded into one displacement at the final grid. Red is the x component (right is positive), green is the y component (down is positive), and gray is no displacement.

Set `CHECKPOINT` to a local `.pth` to skip the Hugging Face download. `MODEL_NAME` must be a key in `CONFIGS`.

The third image is that same composed field in Middlebury optical-flow colors: hue is direction and white is no displacement. Saturation grows with magnitude. The wheel beside it is labeled up, down, left, and right.

Two further strips show that composed field after every block, in the same left-to-right order as the inference montage. Early columns use the same color scale as the last column, so a dimmer step is a smaller displacement.

Another color-wheel strip shows each block by itself. A column there is only that block's displacement, not the accumulation of earlier blocks.

The last strip is the feature map itself: the backbone grid, then the map after each deformation, through the final grid. Nearest upsamples are applied but not drawn. Colors come from one PCA fit on the input map.

A GIF plays the cumulative optical flow beside the feature map, one shared step per frame.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"

import torch
import torchvision
from PIL import Image

from elasticdino.model.elasticdino import ElasticDino
from elasticdino.visualize_deformations import visualize_deformations

torch.set_float32_matmul_precision("medium")

In [2]:
MODEL_NAME = "elasticdino-32-S-no-reg"
CHECKPOINT = None  # e.g. "/path/to/elasticdino-32-S-no-reg.pth"
DINO_REPO = "facebookresearch/dinov2"

IMAGE_PATH = "assets/tree.jpeg"
OUTPUT_PATH = "deformation_visualization.png"
PANEL = 176

In [ ]:
model = ElasticDino.from_pretrained(MODEL_NAME, checkpoint_path=CHECKPOINT, dino_repo=DINO_REPO)
model = model.cuda()
model.eval()

In [ ]:
image = torchvision.transforms.functional.pil_to_tensor(
    Image.open(IMAGE_PATH).convert("RGB").resize((128, 128))
).unsqueeze(0).cuda() / 255.0

Image.open(IMAGE_PATH).convert("RGB").resize((128, 128))

In [ ]:
from IPython.display import Image as DisplayImage, display

montage, outputs = visualize_deformations(
    model,
    image,
    output_path=OUTPUT_PATH,
    panel=PANEL,
)
print(f"Wrote {OUTPUT_PATH} ({montage.width}x{montage.height})")
print(f"Wrote {outputs['composed_rgb_path']}")
print(f"Wrote {outputs['composed_flow_path']}")
print(f"Wrote {outputs['composed_components_steps_path']}")
print(f"Wrote {outputs['composed_flow_steps_path']}")
print(f"Wrote {outputs['flow_per_step_path']}")
print(f"Wrote {outputs['feature_strip_path']}")
print("Feature maps:", ", ".join(outputs["feature_labels"]))
print(f"Wrote {outputs['flow_feature_gif_path']}")
print("Composition steps:", ", ".join(outputs["composition_steps"]))
print(f"Color wheel full saturation {outputs['composed_flow_scale_px']:.2f}px")
print(f"Final features: {tuple(outputs['deformed_features'].shape)}")
print(f"Composed displacement: {tuple(outputs['composed_displacement'].shape)}, full scale {outputs['composed_scale_px']:.2f}px")
for stage_index, stage in enumerate(outputs["sampling_fields"]):
    for block_index, field in enumerate(stage):
        print(f"stage {stage_index} block {block_index}: sampling field {tuple(field.shape)}")
display(montage)
display(outputs["composed_rgb"])
display(outputs["composed_flow"])
display(outputs["composed_components_steps"])
display(outputs["composed_flow_steps"])
display(outputs["flow_per_step"])
display(outputs["feature_strip"])
display(DisplayImage(filename=outputs["flow_feature_gif_path"]))


In [ ]:
# Convert cell to make strips without upsampler steps for the images I already run old strip generation for.
from pathlib import Path

from elasticdino.visualize_deformations import save_gif

ASSETS_DIR = Path("assets")
CONVERT_DIR = Path("convert")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

image_paths = sorted(
    path for path in ASSETS_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)

for image_path in image_paths:
    out_dir = CONVERT_DIR / image_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    tensor = torchvision.transforms.functional.pil_to_tensor(
        Image.open(image_path).convert("RGB").resize((128, 128))
    ).unsqueeze(0).cuda() / 255.0
    _, outputs = visualize_deformations(model, tensor, output_path=None, panel=PANEL)
    outputs["feature_strip"].save(out_dir / "features.png")
    outputs["composed_flow"].save(out_dir / "flow.png")
    outputs["composed_flow_steps"].save(out_dir / "flow_steps.png")
    outputs["flow_per_step"].save(out_dir / "flow_per_step.png")
    save_gif(
        outputs["flow_feature_frames"],
        str(out_dir / "flow_features.gif"),
        outputs["flow_feature_duration_ms"],
    )
    print(f"Wrote {out_dir}")


In [ ]:
import importlib
from pathlib import Path
from PIL import Image as AssetImage

import elasticdino.visualize_deformations as deformation_viz

importlib.reload(deformation_viz)

# Image file names from assets/. A stem such as "tree" also matches tree.jpeg.
FIG_STRIP_NAMES = [
    # "tree.jpeg",
    'bluecar.png',
    'burger.jpeg',
    'car.jpeg',
    'cat.jpeg',
    'dog-blackbg.jpeg',
    'dog.jpeg',
    'dogperson.jpeg',
    'eevee.png',
    'person.jpeg',
    'pikachu.png',
    'redtruck.png',
    'restaurant.jpeg',
    'shop.jpg',
    'sidewalk.jpeg',
    'street.jpg',
    'tiger.jpeg',
    'woman.jpg',
    'bag.jpg',
    'bird2.jpg',
    'couch.jpg',
    'deer.jpg',
    'deer_statue.jpg',
    'eagle.jpg',
    'horses.jpg',
    'husky.jpg',
    'little_dog.jpg',
    'scared_dog.jpg',
    'shield.png',
    'stick.jpg',
    'stingray.jpg',
    'sumo.jpg',
    'telescope.jpg',
    'tractor.png',
    'game_bat.png',
    'broom1.jpeg',
    'broom2.jpeg',
    'broom3.jpeg',
    'lightpost1.jpeg',
    'lightpost2.jpeg',
    'swords.jpeg',
    'tree.jpeg',
]
FIG_STRIPS_DIR = Path("assets/fig_strips")
FIG_STRIPS_DIR.mkdir(parents=True, exist_ok=True)


def _asset_path(name):
    path = Path(name)
    if not path.is_file():
        path = Path("assets") / name
    if path.is_file():
        return path
    matches = [
        candidate for candidate in Path("assets").iterdir()
        if candidate.stem == Path(name).stem and candidate.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"No single asset matches {name!r}")


for name in FIG_STRIP_NAMES:
    path = _asset_path(name)
    photo = AssetImage.open(path).convert("RGB")
    tensor = torchvision.transforms.functional.pil_to_tensor(
        photo.resize((128, 128))
    ).unsqueeze(0).cuda() / 255.0
    with torch.no_grad():
        outputs = model(tensor, return_displacements=True, return_original_features=True)
    feature_row, flow_row, per_step_row = deformation_viz.progression_panels(
        outputs["original_features"],
        outputs["sampling_fields"],
        panel=PANEL,
    )
    pair = deformation_viz.render_strip_figure(photo, [feature_row, flow_row])
    triple = deformation_viz.render_strip_figure(photo, [feature_row, flow_row, per_step_row])
    pair_path = FIG_STRIPS_DIR / f"{path.stem}_2_strips.png"
    triple_path = FIG_STRIPS_DIR / f"{path.stem}_3_strips.png"
    pair.save(pair_path)
    triple.save(triple_path)
    print(f"Wrote {pair_path} and {triple_path}")
